In [32]:
import requests as rq

In [33]:
from pathlib import Path


root_path = Path(".")
mds = root_path.glob("day_*._md")
mds = list(mds)
mds

[PosixPath('day_2._md'), PosixPath('day_1._md')]

In [34]:
import json
import re


def get_eg_link(title):
    params = {
        "sort": "score,DESC",
        "page": 0,
        "size": 10,
        "query": f'"{title}"',
        "f.dateIssued": "[2007 TO *],equals",
        "embed": "thumbnail, item/thumbnail",
    }
    r = rq.get(
        "https://diglib.eg.org/server/api/discover/search/objects",
        params,
    ).json()
    res_list = r["_embedded"]["searchResult"]["_embedded"]["objects"]
    res_object = res_list[0] if len(res_list) > 0 else None
    # print(json.dumps(res_object, indent=2))
    if res_object is None:
        return {}
    link = f"https://diglib.eg.org/items/{res_object['_embedded']['indexableObject']['id']}"
    doi = res_object["_embedded"]["indexableObject"]["handle"]
    authors = res_object["_embedded"]["indexableObject"]["metadata"][
        "dc.contributor.author"
    ]
    abstr = res_object["_embedded"]["indexableObject"]["metadata"].get(
        "dc.description.abstract", []
    )
    thumbnail = res_object["_embedded"]["indexableObject"]["_embedded"]["thumbnail"]["_links"]["content"]["href"]
    author_names = [a["value"] for a in authors]
    return {
        "link": link,
        "doi": doi,
        "authors": author_names,
        "abstr": abstr,
        "thumbnail": thumbnail
    }


resp = get_eg_link("Parameter Space Analysis through Guided Visual Interpolations")

In [35]:
resp

{'link': 'https://diglib.eg.org/items/860c1d60-ddda-49ba-93b2-21f285c4c410',
 'doi': '10.2312/mlvis20261001',
 'authors': ['Kantz, Benedikt',
  'Waldert, Peter',
  'Lengauer, Stefan',
  'Staudinger, Clemens',
  'Schuster, Stefan',
  'Schreck, Tobias'],
 'abstr': [{'value': 'We propose Parameter Space Analysis through Guided Visual Interpolations (ParamInter), a novel tool for high dimensional input parameter space analysis by making interpolation towards optimal parameter sets explorable using guided analytics. The interpolation is accompanied by both small multiples in linked views, and utilizes t-Distributed Stochastic Neighbor Embedding (t-SNE) representations to show an interpolation overview. ParamInter uses a guided exploration loop focusing on the interpolation towards user-specified target parameters from many output parameters. The exploration process is additionally guided through eXplainable Artificial Intelligence (XAI)-based effect suggestions throughout our tool. ParamInt

In [36]:
out_path = root_path
out_path.mkdir(exist_ok=True)
for md in mds:
    out_file = out_path / f"ev_{md.stem}.md"
    processed_content = []
    with open(md, "r") as f:
        content = f.read()
    lines = content.splitlines()
    for line in lines:
        processed_content.append(line)
        if line.startswith("## "):
            title = line[3:]
            print(f"Processing {title}")
            eg_info = get_eg_link(title)
            author_text = (
                ", ".join(eg_info["authors"])
                if "authors" in eg_info
                else "No authors found"
            )
            authors = (
                f"> Authors: {author_text}"
                if "authors" in eg_info
                else "No authors found"
            )
            link = (
                f"\n[EG Link]({eg_info['link']})"
                if "link" in eg_info
                else "No EG link found"
            )
            thumbnail = (
                f"\n![Thumbnail]({eg_info['thumbnail']})"
                if "thumbnail" in eg_info
                else ""
            )
            processed_content.append(authors)
            processed_content.append(link)
            processed_content.append(thumbnail)
    with open(out_file, "w") as f:
        f.write("\n".join(processed_content))

Processing Anchor Flow Maps: Efficient Sampling and Approximation of the Entire Flow Map
Processing How Historians Use Visualization: A Corpus-Backed Taxonomy and Analysis for Cross-Disciplinary Practice
Processing Keynote on Human-in-the-loop AI systems
Processing Opening the Model Building Loop: Explaining the Role of Visual Model Estimation and Validation in Visual Analytics Pipelines
Processing Parameter Space Analysis through Guided Visual Interpolations
Processing Visual Analysis of Semantic Paraphrase Embebdding Projection Stability
Processing Integrating Gridded Glyph Maps and Self-Organizing Maps for Spatiotemporal Analysis
Processing Judging Lines, Ignoring Noise? Human Approach to Outliers in Visual Regression Validation
Processing Visual Analysis of the Influence of the Winter NAO on Moisture Transport to Europe in Ensemble Climate Simulations
Processing Can the visualization of mesoscale eddies using glyphs help oceanographers evaluate changing ocean dynamics?
Processing P